<a href="https://colab.research.google.com/github/murali-marimekala/pg-genai-ml-priv/blob/main/Prompt_Engienering_Sqlite_12_Jul_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sqlite3
import os
from openai import OpenAI
from google.colab import userdata

# -------------------------------
# Load OpenAI API Key
# -------------------------------
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=OPENAI_API_KEY
)

# -------------------------------
# Create SQLite Database
# -------------------------------
conn = sqlite3.connect("sales.db")
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Sales(
    Product TEXT,
    Category TEXT,
    Sales INTEGER,
    Price INTEGER
)
""")

# Delete old data (optional)
cursor.execute("DELETE FROM Sales")

# Insert sample data
cursor.executemany("""
INSERT INTO Sales
VALUES (?,?,?,?)
""",
[
    ("Laptop","Electronics",120,50000),
    ("Mobile","Electronics",250,20000),
    ("Tablet","Electronics",80,30000),
    ("TV","Electronics",60,45000),
    ("Headphones","Accessories",300,3000)
])

conn.commit()

print("Database Created Successfully.\n")

# -------------------------------
# Chat Loop
# -------------------------------

while True:

    question = input("Ask a Question (type 'bye' to exit): ")

    if question.lower() == "bye":
        break

    # Read data from database
    cursor.execute("SELECT * FROM Sales")
    rows = cursor.fetchall()

    # Prompt
    messages = [
        {
            "role":"developer",
            "content":f"""
You are a Sales Data Analyst.

Below is the Sales Database.

{rows}

Answer ONLY using this database.

If the answer is not available,
say "Data not found."

Keep answers short.
"""
        },
        {
            "role":"user",
            "content":question
        }
    ]

    # Call OpenAI
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    print("\nAssistant:")
    print(response.choices[0].message.content)
    print("-"*60)

# Close database
conn.close()

Database Created Successfully.

Ask a Question (type 'bye' to exit): what is capital of australia

Assistant:
Data not found.
------------------------------------------------------------
Ask a Question (type 'bye' to exit): what are current sales

Assistant:
Current sales total 12,510,000.
------------------------------------------------------------
Ask a Question (type 'bye' to exit): exit

Assistant:
Data not found.
------------------------------------------------------------
Ask a Question (type 'bye' to exit): bye


In [5]:
import sqlite3
import os
from openai import OpenAI
from google.colab import userdata

# -----------------------------
# Load OpenAI Key
# -----------------------------
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=OPENAI_API_KEY
)

# -----------------------------
# Create SQLite Database
# -----------------------------
conn = sqlite3.connect("sales.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS Sales(
    Product TEXT,
    Category TEXT,
    Sales INTEGER,
    Price INTEGER
)
""")

cursor.execute("DELETE FROM Sales")

cursor.executemany("""
INSERT INTO Sales VALUES(?,?,?,?)
""",
[
    ("Laptop","Electronics",120,50000),
    ("Mobile","Electronics",250,20000),
    ("Tablet","Electronics",80,30000),
    ("TV","Electronics",60,45000),
    ("Headphones","Accessories",300,3000)
])

conn.commit()

print("Database Created Successfully")

# -----------------------------
# Database Schema
# -----------------------------

schema = """
Table Name : Sales

Columns:

Product TEXT
Category TEXT
Sales INTEGER
Price INTEGER
"""

# -----------------------------
# Chat Loop
# -----------------------------

while True:

    question = input("\nAsk Question (bye to exit): ")

    if question.lower() == "bye":
        break

    # =====================================================
    # STEP 1 : Ask GPT to Generate SQL
    # =====================================================

    sql_prompt = [
        {
            "role":"developer",
            "content":f"""
You are an SQLite expert.

Database Schema:

{schema}

Rules:

1. Generate ONLY SQLite SQL.
2. Do not explain.
3. Do not use markdown.
4. Return only SQL.
"""
        },
        {
            "role":"user",
            "content":question
        }
    ]

    sql_response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=sql_prompt
    )

    sql_query = sql_response.choices[0].message.content.strip()

    print("\nGenerated SQL:")
    print(sql_query)

    # =====================================================
    # STEP 2 : Execute SQL
    # =====================================================

    try:

        cursor.execute(sql_query)

        result = cursor.fetchall()

        print("\nSQL Result:")
        print(result)

    except Exception as e:

        print(e)
        continue

    # =====================================================
    # STEP 3 : Ask GPT to Explain Result
    # =====================================================

    answer_prompt = [
        {
            "role":"developer",
            "content":"""
You are a Sales Assistant.

Answer using ONLY the SQL result.

If SQL result is empty,
say Data not found.

Keep answer within two sentences.
"""
        },
        {
            "role":"user",
            "content":f"""
Question:

{question}

SQL Result:

{result}
"""
        }
    ]

    answer = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=answer_prompt
    )

    print("\nAssistant:")
    print(answer.choices[0].message.content)

conn.close()

Database Created Successfully

Ask Question (bye to exit): what is the price of laptop 

Generated SQL:
SELECT Price FROM Sales WHERE Product = 'laptop';

SQL Result:
[]

Assistant:
Data not found.

Ask Question (bye to exit): vye

Generated SQL:
SELECT * FROM Sales;

SQL Result:
[('Laptop', 'Electronics', 120, 50000), ('Mobile', 'Electronics', 250, 20000), ('Tablet', 'Electronics', 80, 30000), ('TV', 'Electronics', 60, 45000), ('Headphones', 'Accessories', 300, 3000)]

Assistant:
We have the following products available: Laptop, Mobile, Tablet, TV, and Headphones. Laptops have 120 units at 50,000 each, Mobiles have 250 units at 20,000 each, Tablets have 80 units at 30,000, TVs have 60 units at 45,000, and Headphones have 300 units at 3,000.

Ask Question (bye to exit): bye
